<font color=skyblue>**Full windowed GUI-based deblurring app version 1**</font>

This notebook cell contains a full windowed GUI-based deblurring app, which supports repeated runs without re-executing the notebook cell. The app allows users to select an image file, apply a deblurring model, and display the results in a user-friendly interface. The app is designed to be efficient and responsive, providing a seamless experience for users who want to deblur their images.

In [6]:
import os
import cv2
import torch
import numpy as np
import tkinter as tk
import threading
from torchvision import transforms
from tkinter import filedialog, messagebox, ttk
from PIL import Image, ImageTk
from Deblurring_defs import (
    DeblurDataset, DeblurCNN, DeblurCNN_RES, DeblurSuperResCNN, NAFNet, psnr
)

# Full windowed GUI-based deblurring app.
# It supports repeated runs without re-executing the notebook cell.
device = 'cpu'
pre_trained_model = '../outputs/pre_trained_DeblurCNN_patch_100.pt'

def load_deblur_model(weights_path, device='cpu'):
    checkpoint = torch.load(weights_path, map_location=device) if os.path.exists(weights_path) else None
    if checkpoint is None:
        raise FileNotFoundError(f'No pre-trained model found at: {weights_path}')

    saved_model_class = checkpoint.get('model_class', 'DeblurCNN')
    saved_model_init_args = checkpoint.get('model_init_args', {})

    if saved_model_class == 'DeblurCNN':
        model = DeblurCNN(**saved_model_init_args).to(device)
    elif saved_model_class == 'DeblurCNN_RES':
        model = DeblurCNN_RES(**saved_model_init_args).to(device)
    elif saved_model_class == 'DeblurSuperResCNN':
        model = DeblurSuperResCNN(**saved_model_init_args).to(device)
    elif saved_model_class == 'NAFNet':
        model = NAFNet(**saved_model_init_args).to(device)
    else:
        print(f"Unknown saved model class '{saved_model_class}', fallback to DeblurCNN().")
        model = DeblurCNN().to(device)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

def get_recent_model_candidates(default_model, search_dir='../outputs', limit=8):
    candidates = []

    if default_model:
        candidates.append(default_model)

    if os.path.isdir(search_dir):
        model_files = []
        for name in os.listdir(search_dir):
            low = name.lower()
            if low.endswith('.pt') or low.endswith('.pth'):
                full_path = os.path.join(search_dir, name)
                try:
                    mtime = os.path.getmtime(full_path)
                except OSError:
                    mtime = 0.0
                model_files.append((mtime, full_path))

        model_files.sort(key=lambda x: x[0], reverse=True)
        for _, path in model_files[:limit * 2]:
            candidates.append(path)

    unique = []
    seen = set()
    for p in candidates:
        if not p:
            continue
        norm = os.path.normpath(p)
        if norm not in seen:
            seen.add(norm)
            unique.append(p)
        if len(unique) >= limit:
            break

    return unique

class DeblurApp:
    def __init__(self, root, model=None, checkpoint=None, model_path='', recent_models=None, device='cpu'):
        self.root = root
        self.model = model
        self.checkpoint = checkpoint if checkpoint is not None else {}
        self.device = device
        self.model_path = model_path
        self.recent_models = list(recent_models) if recent_models else []

        self.root.title('Image Deblurring GUI')
        self.root.geometry('1300x760')

        self.image_path = None
        self.sharp_np = None
        self.blur_np = None
        self.deblur_np = None

        self.sharp_tk = None
        self.blur_tk = None
        self.deblur_tk = None

        self.sigma_var = tk.DoubleVar(value=3.0)
        self.status_var = tk.StringVar(value='Select model and image, then click Run Deblur.')
        self.recent_model_var = tk.StringVar(value='')

        model_name = self.checkpoint.get('model_class', 'Not loaded')
        model_epoch = self.checkpoint.get('epoch', 0)

        top_frame = tk.Frame(self.root)
        top_frame.pack(fill='x', padx=12, pady=(10, 4))

        self.model_info_label = tk.Label(
            top_frame,
            text=f'Model: {model_name}    Epochs trained: {model_epoch}',
            font=('Segoe UI', 11, 'bold')
        )
        self.model_info_label.pack(anchor='w')

        self.model_path_var = tk.StringVar(
            value=f'Weights: {self.model_path if self.model_path else "(not selected)"}'
        )
        self.model_path_label = tk.Label(
            top_frame,
            textvariable=self.model_path_var,
            anchor='w',
            fg='gray30'
        )
        self.model_path_label.pack(anchor='w')

        recent_frame = tk.Frame(top_frame)
        recent_frame.pack(fill='x', pady=(6, 0))

        tk.Label(recent_frame, text='Recent models:').pack(side='left', padx=(0, 6))

        self.recent_combo = ttk.Combobox(
            recent_frame,
            textvariable=self.recent_model_var,
            state='readonly',
            width=95
        )
        self.recent_combo.pack(side='left', padx=(0, 8), fill='x', expand=True)

        self.load_recent_btn = tk.Button(
            recent_frame, text='Load Recent', width=12, command=self.load_selected_recent_model
        )
        self.load_recent_btn.pack(side='left')

        control_frame = tk.Frame(self.root)
        control_frame.pack(fill='x', padx=12, pady=6)

        self.select_model_btn = tk.Button(
            control_frame, text='0) Select Model', width=16, command=self.select_model
        )
        self.select_model_btn.grid(row=0, column=0, padx=(0, 8), pady=4)

        self.select_btn = tk.Button(
            control_frame, text='1) Select Image', width=16, command=self.select_image
        )
        self.select_btn.grid(row=0, column=1, padx=(0, 8), pady=4)

        self.path_label = tk.Label(
            control_frame, text='No image selected', anchor='w', width=85
        )
        self.path_label.grid(row=0, column=2, columnspan=4, sticky='w', padx=4)

        sigma_text = tk.Label(control_frame, text='SigmaX:')
        sigma_text.grid(row=1, column=0, sticky='e', padx=(0, 4), pady=4)

        self.sigma_value_label = tk.Label(control_frame, text='3.0', width=5)
        self.sigma_value_label.grid(row=1, column=1, sticky='w', padx=(0, 6), pady=4)

        self.sigma_slider = tk.Scale(
            control_frame,
            from_=0.5,
            to=10.0,
            resolution=0.1,
            orient='horizontal',
            length=320,
            variable=self.sigma_var,
            command=self.on_sigma_change
        )
        self.sigma_slider.grid(row=1, column=2, sticky='w', padx=4, pady=4)

        self.run_btn = tk.Button(
            control_frame, text='2) Run Deblur', width=16, command=self.run_deblur
        )
        self.run_btn.grid(row=1, column=3, padx=8, pady=4)

        self.save_btn = tk.Button(
            control_frame,
            text='3) Save Deblurred',
            width=16,
            command=self.save_deblurred,
            state='disabled'
        )
        self.save_btn.grid(row=1, column=4, padx=8, pady=4)

        self.quit_btn = tk.Button(
            control_frame, text='Exit', width=10, command=self.root.destroy
        )
        self.quit_btn.grid(row=1, column=5, padx=(8, 0), pady=4)

        preview_frame = tk.Frame(self.root)
        preview_frame.pack(fill='both', expand=True, padx=12, pady=8)

        left_panel = tk.Frame(preview_frame, bd=1, relief='solid')
        mid_panel = tk.Frame(preview_frame, bd=1, relief='solid')
        right_panel = tk.Frame(preview_frame, bd=1, relief='solid')
        left_panel.pack(side='left', fill='both', expand=True, padx=4)
        mid_panel.pack(side='left', fill='both', expand=True, padx=4)
        right_panel.pack(side='left', fill='both', expand=True, padx=4)

        self.left_title = tk.Label(left_panel, text='Blurred Image', font=('Segoe UI', 10, 'bold'))
        self.left_title.pack(pady=(8, 4))
        self.mid_title = tk.Label(mid_panel, text='Deblurred Image', font=('Segoe UI', 10, 'bold'))
        self.mid_title.pack(pady=(8, 4))
        self.right_title = tk.Label(right_panel, text='Sharp Image', font=('Segoe UI', 10, 'bold'))
        self.right_title.pack(pady=(8, 4))

        self.left_img_label = tk.Label(left_panel, text='No preview')
        self.left_img_label.pack(fill='both', expand=True, padx=8, pady=(0, 8))

        self.mid_img_label = tk.Label(mid_panel, text='No preview')
        self.mid_img_label.pack(fill='both', expand=True, padx=8, pady=(0, 8))

        self.right_img_label = tk.Label(right_panel, text='No preview')
        self.right_img_label.pack(fill='both', expand=True, padx=8, pady=(0, 8))

        status_label = tk.Label(
            self.root, textvariable=self.status_var, anchor='w', fg='navy'
        )
        status_label.pack(fill='x', padx=12, pady=(0, 10))

        self.refresh_recent_dropdown()
        if self.model_path:
            self.add_recent_model(self.model_path, make_current=True)

    def on_sigma_change(self, _event=None):
        self.sigma_value_label.config(text=f'{self.sigma_var.get():.1f}')

    def refresh_recent_dropdown(self):
        if self.recent_models:
            self.recent_combo['values'] = self.recent_models
            if self.recent_model_var.get() not in self.recent_models:
                self.recent_model_var.set(self.recent_models[0])
            self.recent_combo.config(state='readonly')
            self.load_recent_btn.config(state='normal')
        else:
            self.recent_combo['values'] = ['(no recent models)']
            self.recent_model_var.set('(no recent models)')
            self.recent_combo.config(state='disabled')
            self.load_recent_btn.config(state='disabled')

    def add_recent_model(self, model_path, make_current=True):
        if not model_path:
            return

        norm_path = os.path.normpath(model_path)
        dedup = []
        seen = set()

        dedup.append(norm_path)
        seen.add(norm_path)

        for existing in self.recent_models:
            norm_existing = os.path.normpath(existing)
            if norm_existing not in seen:
                dedup.append(norm_existing)
                seen.add(norm_existing)

        self.recent_models = dedup[:10]
        self.refresh_recent_dropdown()

        if make_current and self.recent_models:
            self.recent_model_var.set(self.recent_models[0])

    def _apply_loaded_model(self, model, checkpoint, model_path):
        self.model = model
        self.checkpoint = checkpoint
        self.model_path = model_path

        model_name = self.checkpoint.get('model_class', 'DeblurCNN')
        model_epoch = self.checkpoint.get('epoch', 0)
        self.model_info_label.config(text=f'Model: {model_name}    Epochs trained: {model_epoch}')
        self.model_path_var.set(f'Weights: {model_path}')
        self.add_recent_model(model_path, make_current=True)
        self.status_var.set('Model loaded. Select image and run deblur.')

    def _load_model_from_path(self, model_path):
        model, checkpoint = load_deblur_model(model_path, device=self.device)
        self._apply_loaded_model(model, checkpoint, model_path)

    def load_selected_recent_model(self):
        selected = self.recent_model_var.get()
        if not selected or selected == '(no recent models)':
            return

        if not os.path.exists(selected):
            messagebox.showerror('Model Load Error', f'File does not exist:\n{selected}')
            return

        try:
            self._load_model_from_path(selected)
        except Exception as e:
            messagebox.showerror('Model Load Error', f'Failed to load model:\n{e}')

    def select_model(self):
        model_path = filedialog.askopenfilename(
            title='Select pre-trained model',
            filetypes=[('Model Files', '*.pt *.pth'), ('All Files', '*.*')]
        )

        if not model_path:
            return

        try:
            self._load_model_from_path(model_path)
        except Exception as e:
            messagebox.showerror('Model Load Error', f'Failed to load model:\n{e}')

    def select_image(self):
        img_path = filedialog.askopenfilename(
            title='Select an image to deblur',
            filetypes=[('Image Files', '*.png *.jpg *.jpeg *.bmp *.tif *.tiff')]
        )

        if not img_path:
            return

        sharp = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if sharp is None:
            messagebox.showerror('Error', f'Failed to read image:\n{img_path}')
            return

        self.image_path = img_path
        self.sharp_np = cv2.cvtColor(sharp, cv2.COLOR_BGR2RGB)
        self.blur_np = None
        self.deblur_np = None

        self.path_label.config(text=img_path)
        self.right_title.config(text='Sharp Image')
        self.update_image_label(self.right_img_label, self.sharp_np, 'sharp')
        self.left_img_label.config(image='', text='Run deblur to preview')
        self.mid_img_label.config(image='', text='Run deblur to preview')
        self.left_title.config(text='Blurred Image')
        self.mid_title.config(text='Deblurred Image')
        self.save_btn.config(state='disabled')
        self.status_var.set('Image loaded. Adjust sigma and click Run Deblur.')

    def run_deblur(self):
        if self.model is None:
            messagebox.showwarning('Warning', 'Please select a pre-trained model first.')
            return

        if self.sharp_np is None:
            messagebox.showwarning('Warning', 'Please select an image first.')
            return

        sigmaX = float(self.sigma_var.get())
        self.blur_np = cv2.GaussianBlur(self.sharp_np, (0, 0), sigmaX=sigmaX)

        sharp_tensor = transforms.ToTensor()(self.sharp_np).unsqueeze(0).to(self.device)
        blur_tensor = transforms.ToTensor()(self.blur_np).unsqueeze(0).to(self.device)

        with torch.no_grad():
            deblurred_tensor = self.model(blur_tensor)

        blur_eval = blur_tensor.clamp(0, 1)
        deblur_eval = deblurred_tensor.clamp(0, 1)

        psnr_blur = psnr(sharp_tensor, blur_eval)
        psnr_deblur = psnr(sharp_tensor, deblur_eval)

        self.deblur_np = np.clip(
            deblur_eval.squeeze(0).cpu().permute(1, 2, 0).numpy(), 0, 1
        )
        blur_show = np.clip(
            blur_eval.squeeze(0).cpu().permute(1, 2, 0).numpy(), 0, 1
        )

        self.left_title.config(text=f'Blurred Image (PSNR: {psnr_blur:.2f} dB)')
        self.mid_title.config(text=f'Deblurred Image (PSNR: {psnr_deblur:.2f} dB)')
        self.right_title.config(text='Sharp Image')

        self.update_image_label(self.left_img_label, blur_show, 'blur')
        self.update_image_label(self.mid_img_label, self.deblur_np, 'deblur')
        self.update_image_label(self.right_img_label, self.sharp_np, 'sharp')

        self.save_btn.config(state='normal')
        self.status_var.set(
            f'Deblur complete at sigmaX={sigmaX:.1f}. You can change sigma and run again.'
        )

    def save_deblurred(self):
        if self.deblur_np is None:
            messagebox.showwarning('Warning', 'No deblurred image to save. Run deblur first.')
            return

        initial_name = 'deblurred_result.png'
        if self.image_path is not None:
            base = os.path.splitext(os.path.basename(self.image_path))[0]
            initial_name = f'{base}_deblurred.png'

        save_path = filedialog.asksaveasfilename(
            title='Save deblurred image',
            defaultextension='.png',
            initialfile=initial_name,
            filetypes=[
                ('PNG', '*.png'),
                ('JPEG', '*.jpg *.jpeg'),
                ('Bitmap', '*.bmp'),
                ('TIFF', '*.tif *.tiff')
            ]
        )

        if not save_path:
            return

        img_to_save = (np.clip(self.deblur_np, 0, 1) * 255).astype(np.uint8)
        img_to_save_bgr = cv2.cvtColor(img_to_save, cv2.COLOR_RGB2BGR)
        ok = cv2.imwrite(save_path, img_to_save_bgr)

        if ok:
            self.status_var.set(f'Saved deblurred image: {save_path}')
        else:
            messagebox.showerror('Error', f'Failed to save image:\n{save_path}')

    def update_image_label(self, label_widget, np_img, image_slot):
        display = np_img
        if display.dtype != np.uint8:
            display = (np.clip(display, 0, 1) * 255).astype(np.uint8)

        h, w = display.shape[:2]
        max_w, max_h = 380, 420
        scale = min(max_w / max(1, w), max_h / max(1, h), 1.0)
        new_w = max(1, int(w * scale))
        new_h = max(1, int(h * scale))

        resized = cv2.resize(display, (new_w, new_h), interpolation=cv2.INTER_AREA)
        pil_img = Image.fromarray(resized)
        tk_img = ImageTk.PhotoImage(pil_img)

        label_widget.config(image=tk_img, text='')

        if image_slot == 'sharp':
            self.sharp_tk = tk_img
        elif image_slot == 'blur':
            self.blur_tk = tk_img
        elif image_slot == 'deblur':
            self.deblur_tk = tk_img

def _run_gui_thread():
    model, checkpoint, model_path = None, {}, ''
    recent_models = get_recent_model_candidates(pre_trained_model, search_dir='../outputs', limit=8)

    if pre_trained_model and os.path.exists(pre_trained_model):
        try:
            model, checkpoint = load_deblur_model(pre_trained_model, device=device)
            model_path = pre_trained_model
        except Exception as e:
            print(f'Could not auto-load default model: {e}')
            print('Use Select Model or Recent Models to choose a .pt/.pth file.')
    else:
        print('Default model not found. Use Select Model or Recent Models.')

    root = tk.Tk()
    app = DeblurApp(
        root,
        model=model,
        checkpoint=checkpoint,
        model_path=model_path,
        recent_models=recent_models,
        device=device
    )
    root.mainloop()

def launch_deblur_gui(blocking=False):
    if blocking:
        _run_gui_thread()
        return None

    gui_thread = threading.Thread(target=_run_gui_thread, daemon=True)
    gui_thread.start()
    print('Deblur GUI launched in background. This cell can finish immediately.')
    print('Use Select Model or the Recent Models dropdown to load .pt/.pth files.')
    print('If no window appears, check whether it opened behind other windows.')
    return gui_thread

GUI_THREAD = launch_deblur_gui(blocking=False)

Deblur GUI launched in background. This cell can finish immediately.
Use Select Model or the Recent Models dropdown to load .pt/.pth files.
If no window appears, check whether it opened behind other windows.
